# Notebook 1 - Transkribus Export

This notebook exports the already digitized Magazine Issues of Heresies Magazine.

I used the acdh-transkribus-utils Python library (MIT License) to download and export XML from the Transkribus REST API. The original MIT license and copyright notice are preserved.

### Sources
- Austrian Centre for Digital Humanities. (2025). Acdh-oeaw/acdh-transkribus-utils [Python]. https://github.com/acdh-oeaw/acdh-transkribus-utils (Original work published 2020)
- Acdh-transkribus-utils/transkribus_utils/transkribus_utils.py at main · acdh-oeaw/acdh-transkribus-utils. (n.d.). GitHub. Retrieved May 28, 2026, from https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/transkribus_utils.py


### 1. Imports and more

In [8]:
from pathlib import Path

DATA_BASE = Path.home() / "/Users/sophiehamann/master-thesis-code/data"

RAW_DIR = DATA_BASE / "raw"
PROCESSED_DIR = DATA_BASE / "processed"

RAW_TRANSKRIBUS_XML = RAW_DIR / "METS"
RAW_PAGE_XML = RAW_DIR / "transkribus" / "page_xml"
RAW_PDF = RAW_DIR / "pdf" / "issues"

PROCESSED_TEI = PROCESSED_DIR / "tei"
PROCESSED_GT = PROCESSED_DIR / "training_data" / "ground_truth"

In [ ]:
# making sure this directory structure exists
for d in [
    RAW_TRANSKRIBUS_XML,
    RAW_PAGE_XML,
    PROCESSED_TEI,
    PROCESSED_GT,
]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
import os

# I used this documentation: https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/__init__.py

from transkribus_utils.transkribus_utils import ACDHTranskribusUtils


tr_user = os.environ.get("TRANSKRIBUS_USER")
tr_pw = os.environ.get("TRANSKRIBUS_PASSWORD")

client = ACDHTranskribusUtils(user=tr_user, password=tr_pw)

### 2. List all Collections

In [ ]:
collections = client.list_collections()
for x in collections[-7:]:
    print(x["colId"], x["colName"])

2204941 heresies_collection
2222244 heresies_training_dataset
2257154 Digitizing_Heresies
2291396 baseline_training_data
2297059 baseline_test
2324844 digitizing hard stuff
2380042 Jstor_source_heresies


I am working with "2204941 heresies_collection".

The needed ID is 2204941.

In [ ]:
col_id = 2204941          # heresies collection
documents = client.list_docs(col_id) # get all documents in collection

https://transkribus.eu/TrpServer/rest/collections/2204941/list


In [15]:
# inspecting 2204941 heresies_collection 
print(type(documents))
print(len(documents))

# sorting documents by title
documents = sorted(documents, key=lambda d: d["title"])
[d["title"] for d in documents]

<class 'list'>
27


['heresies_01',
 'heresies_02',
 'heresies_03',
 'heresies_04',
 'heresies_05',
 'heresies_06',
 'heresies_07',
 'heresies_08',
 'heresies_09',
 'heresies_10',
 'heresies_11',
 'heresies_12',
 'heresies_13',
 'heresies_14',
 'heresies_15',
 'heresies_16',
 'heresies_17',
 'heresies_18',
 'heresies_19',
 'heresies_20',
 'heresies_21',
 'heresies_22',
 'heresies_23',
 'heresies_24',
 'heresies_25',
 'heresies_26',
 'heresies_27']

In [ ]:
# creating a look up to find the corresponding docId for a given document title
doc_map = {d["title"]: d["docId"] for d in documents}


# Get all available docIds for download
available_doc_ids = list(doc_map.values())
print(available_doc_ids)

[11509115, 11501518, 11501519, 11501559, 11501520, 11501560, 11501521, 11501561, 11501522, 11501562, 11501563, 11501564, 11501523, 11501565, 15401067, 11501567, 11501568, 11501569, 11501570, 11501571, 16016338, 16016359, 16016635, 11501573, 16018718, 16018836, 11501783]


### 3. List all documents from the given collection (heresies_collection)

In [ ]:
# checking what metadata fields are available in the collection
for k in documents[-3:]:
    print(k.keys())

dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'storeType', 'storeRegion', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])
dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'storeType', 'storeRegion', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])
dict_keys(['type', 'docId', 'title', 'uploadTimestamp', 'uploader', 'uploaderId', 'nrOfPages', 'pageId', 'url', 'thumbUrl', 'status', 'fimgStoreColl', 'storeType', 'origDocId', 'collectionList', 'attributes', 'labels', 'pageLabels', 'mainColId'])


In [ ]:
# exploratory inspection of document metadata
documents = client.list_docs(col_id)

for doc in documents:
    print(
        doc["docId"],
        doc["title"],
        doc["nrOfPages"]
    )

https://transkribus.eu/TrpServer/rest/collections/2204941/list
11501518 heresies_02 128
11501519 heresies_03 122
11501520 heresies_05 139
11501521 heresies_07 130
11501522 heresies_09 100
11501523 heresies_13 100
11501559 heresies_04 132
11501560 heresies_06 132
11501561 heresies_08 132
11501562 heresies_10 100
11501563 heresies_11 98
11501564 heresies_12 100
11501565 heresies_14 42
11501567 heresies_16 98
11501568 heresies_17 100
11501569 heresies_18 97
11501570 heresies_19 33
11501571 heresies_20 98
11501573 heresies_24 42
11501783 heresies_27 120
11509115 heresies_01 116
15401067 heresies_15 76
16016338 heresies_21 100
16016359 heresies_22 100
16016635 heresies_23 101
16018718 heresies_25 116
16018836 heresies_26 112


### 4. Inspect one Document from the Given Collection
This part of the code inspects a single document out of the Transkribus heresies_collection.
I am looking at document metadata, page lists, page xml and transcripts.

All Python code used in this thesis for accessing, inspecting, and exporting Transkribus documents is based exclusively on the publicly available and MIT-licensed acdh-transkribus-utils library, without introducing additional custom functionality beyond the methods provided in the original repository.
[see here](https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/transkribus_utils.py)

In [21]:
# defining document ID
doc_id_1 = 11509115         # heresies_01

In [22]:
# inspecting document metadata of Heresies 01
doc_metadata_1 = client.get_doc_md(doc_id_1, col_id)
print(doc_metadata_1)    

{'type': 'trpDocMetadata', 'docId': 11509115, 'title': 'heresies_01', 'uploadTimestamp': 1760968484613, 'uploader': 'a12248219@unet.univie.ac.at', 'uploaderId': 447621, 'nrOfPages': 116, 'pageId': 104610900, 'url': 'https://files.transkribus.eu/Get?fileType=view&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'thumbUrl': 'https://files.transkribus.eu/Get?fileType=thumb&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'status': 0, 'fimgStoreColl': 'TrpDoc_DEA_11509115', 'storeType': 0, 'origDocId': 0, 'collectionList': {'colList': [{'colId': 2204941, 'colName': 'heresies_collection', 'description': 'created by a12248219@unet.univie.ac.at', 'crowdsourcing': False, 'elearning': False, 'nrOfDocuments': 0, 'isFavorite': False}]}, 'attributes': [], 'labels': [{'labelId': 127, 'name': 'Ready for Export', 'color': '#1E90FF'}], 'pageLabels': [], 'mainColId': 2204941}


In [ ]:
# inspecting document overview in markdown format
overview = client.get_doc_overview_md(doc_id_1, col_id)
print(overview["trp_return"]["md"])

{'nrOfRegions': 1029, 'nrOfTranscribedRegions': 0, 'nrOfWordsInRegions': 0, 'nrOfLines': 8712, 'nrOfTranscribedLines': 8706, 'nrOfWordsInLines': 63655, 'nrOfWords': 0, 'nrOfTranscribedWords': 0, 'nrOfCharsInLines': 305727, 'nrOfTables': 0, 'nrOfNew': 0, 'nrOfInProgress': 0, 'nrOfDone': 0, 'nrOfFinal': 116, 'nrOfGT': 0, 'docId': 11509115, 'title': 'heresies_01', 'uploadTimestamp': 1760968484613, 'uploader': 'a12248219@unet.univie.ac.at', 'uploaderId': 447621, 'nrOfPages': 116, 'pageId': 104610900, 'url': 'https://files.transkribus.eu/Get?fileType=view&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'thumbUrl': 'https://files.transkribus.eu/Get?fileType=thumb&id=NWEWFYMAIPUDDPYUUBMMLBCD', 'status': 0, 'fimgStoreColl': 'TrpDoc_DEA_11509115', 'storeType': 0, 'origDocId': 0, 'collectionList': {'colList': [{'colId': 2204941, 'colName': 'heresies_collection', 'description': 'created by a12248219@unet.univie.ac.at', 'crowdsourcing': False, 'elearning': False, 'nrOfDocuments': 0, 'isFavorite': False}]}, 'attribu

In [26]:
#inspecting full xml
fulldoc = client.get_fulldoc_md(doc_id_1, col_id, page_id=5)
print(fulldoc["doc_xml"])

fulldoc = client.get_transcript(fulldoc)
print(fulldoc["transcript"][:10])

<Element trpPage at 0x10fd38bc0>
['3', 'the same power and the same intent, and in-', 'dicating that word and image can be equal', 'ingredients in politically effective art.', 'We found no solutions to the issues raised,', 'but we are finding approaches that feel fresher', 'and more satisfying. Working together toward', 'collective decisions was entirely different from', 'working alone or as part of conventional hier-', 'archies. Each of us worked on every page of this']


In [28]:
# I want to see the xml structure of a page
from lxml import etree

print(etree.tostring(fulldoc["doc_xml"], pretty_print=True).decode())

<trpPage>
  <pageId>104610902</pageId>
  <docId>11509115</docId>
  <pageNr>5</pageNr>
  <key>AUWVBEUKXASYBYSDIASPIETA</key>
  <imageId>87781844</imageId>
  <url>https://files.transkribus.eu/Get?fileType=view&amp;id=AUWVBEUKXASYBYSDIASPIETA</url>
  <thumbUrl>https://files.transkribus.eu/Get?fileType=thumb&amp;id=AUWVBEUKXASYBYSDIASPIETA</thumbUrl>
  <storeId>1</storeId>
  <md5Sum>57dfca623b101345334a38ac5f2d1e4f</md5Sum>
  <fileSize>1421330</fileSize>
  <imgFileName>p005.jpg</imgFileName>
  <hideOnSites>false</hideOnSites>
  <tsList>
    <transcripts>
      <tsId>288431270</tsId>
      <parentTsId>-1</parentTsId>
      <storeId>1</storeId>
      <key>RVYZWZAZEETTCFWATOXTIXVX</key>
      <pageId>104610902</pageId>
      <docId>11509115</docId>
      <pageNr>5</pageNr>
      <url>https://files.transkribus.eu/Get?id=RVYZWZAZEETTCFWATOXTIXVX</url>
      <status>FINAL</status>
      <userName>a12248219@unet.univie.ac.at</userName>
      <userId>447621</userId>
      <timestamp>1779443471466<

### 5. Download METS files from Collection
The METS files exported from Transkribus serve as structural containers and reference PAGE-XML files, in which the actual transcribed text is encoded within <TextLine> and <Unicode> elements.

In [9]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils


client.collection_to_mets(
    col_id,
    file_path=str(RAW_TRANSKRIBUS_XML)
)

https://transkribus.eu/TrpServer/rest/collections/2204941/list
27 to download
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501518_mets.xml
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501518_image_name.xml
1/27
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501519_mets.xml
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501519_image_name.xml
2/27
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501520_mets.xml
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501520_image_name.xml
3/27
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501521_mets.xml
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501521_image_name.xml
4/27
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501522_mets.xml
saving: /Users/sophiehamann/master-thesis-code/data/raw/METS/2204941/11501522_image_name.xml
5/

[11501518,
 11501519,
 11501520,
 11501521,
 11501522,
 11501523,
 11501559,
 11501560,
 11501561,
 11501562,
 11501563,
 11501564,
 11501565,
 11501567,
 11501568,
 11501569,
 11501570,
 11501571,
 11501573,
 11501783,
 11509115,
 15401067,
 16016338,
 16016359,
 16016635,
 16018718,
 16018836]

### 6. Download PAGE-XML
I have used [this official Transkribus documentation for the following part](https://github.com/acdh-oeaw/acdh-transkribus-utils/blob/main/transkribus_utils/transkribus_utils.py).

In [37]:
from transkribus_utils.transkribus_utils import ACDHTranskribusUtils
from lxml import etree
import os

client = ACDHTranskribusUtils()

doc_id = 11509115  # heresies_01

In [39]:
overview = client.get_doc_overview_md(doc_id, col_id)
pages = overview["pages"]

#### 6.1 Downloading one file per page for all available documents

In [42]:
print(f"Found {len(doc_map)}")
print(doc_map)

for title, doc_id in doc_map.items():
    # I am creating a directory for each issue to store the page xml files
    output_dir = RAW_DIR / f"{title}_pagexml"
    os.makedirs(output_dir, exist_ok=True)
    
    # getting pages for this document
    overview = client.get_doc_overview_md(doc_id, col_id)
    pages = overview["pages"]
    
    # downloading each page
    for p in pages:
        page_id = p["page_nr"]

        fulldoc = client.get_fulldoc_md(doc_id, col_id, page_id=page_id)
        fulldoc = client.get_transcript(fulldoc)

        page_xml = fulldoc["page_xml"]

        out_file = output_dir / f"page_{page_id}.xml"
        with open(out_file, "wb") as f:
            f.write(etree.tostring(page_xml, pretty_print=True))

Found 27
{'heresies_01': 11509115, 'heresies_02': 11501518, 'heresies_03': 11501519, 'heresies_04': 11501559, 'heresies_05': 11501520, 'heresies_06': 11501560, 'heresies_07': 11501521, 'heresies_08': 11501561, 'heresies_09': 11501522, 'heresies_10': 11501562, 'heresies_11': 11501563, 'heresies_12': 11501564, 'heresies_13': 11501523, 'heresies_14': 11501565, 'heresies_15': 15401067, 'heresies_16': 11501567, 'heresies_17': 11501568, 'heresies_18': 11501569, 'heresies_19': 11501570, 'heresies_20': 11501571, 'heresies_21': 16016338, 'heresies_22': 16016359, 'heresies_23': 16016635, 'heresies_24': 11501573, 'heresies_25': 16018718, 'heresies_26': 16018836, 'heresies_27': 11501783}


#### 6.2 Combining each issue into one XML file

In [45]:
for issue_num in range(1, 28):
    issue_name = f"heresies_{issue_num:02d}"
    input_dir = Path(f"/Users/sophiehamann/master-thesis-code/data/raw/{issue_name}_pagexml")
    
    if not input_dir.exists():
        print(f"Skipping {issue_name} - directory not found")
        continue
    
    root = etree.Element("Document")
    
    # I want to have all XML files in the directory and sorted by page number
    xml_files = sorted(input_dir.glob("page_*.xml"), 
                      key=lambda x: int(x.stem.split('_')[1]))
    
    for xml_file in xml_files:
        try:
            tree = etree.parse(str(xml_file))
            page_xml = tree.getroot()
            root.append(page_xml)
        except Exception as e:
            print(f"  Error processing {xml_file.name}: {e}")
            continue
    
    # Writing the combined XML file
    output_file = RAW_DIR / f"{issue_name}_combined.xml"
    with open(output_file, "wb") as f:
        f.write(etree.tostring(root, pretty_print=True))
    
    print(f"Saved {issue_name}_combined.xml ({len(xml_files)} pages)\n")

Saved heresies_01_combined.xml (116 pages)

Saved heresies_02_combined.xml (128 pages)

Saved heresies_03_combined.xml (122 pages)

Saved heresies_04_combined.xml (132 pages)

Saved heresies_05_combined.xml (139 pages)

Saved heresies_06_combined.xml (132 pages)

Saved heresies_07_combined.xml (130 pages)

Saved heresies_08_combined.xml (132 pages)

Saved heresies_09_combined.xml (100 pages)

Saved heresies_10_combined.xml (100 pages)

Saved heresies_11_combined.xml (98 pages)

Saved heresies_12_combined.xml (100 pages)

Saved heresies_13_combined.xml (100 pages)

Saved heresies_14_combined.xml (42 pages)

Saved heresies_15_combined.xml (76 pages)

Saved heresies_16_combined.xml (98 pages)

Saved heresies_17_combined.xml (100 pages)

Saved heresies_18_combined.xml (97 pages)

Saved heresies_19_combined.xml (33 pages)

Saved heresies_20_combined.xml (98 pages)

Saved heresies_21_combined.xml (100 pages)

Saved heresies_22_combined.xml (100 pages)

Saved heresies_23_combined.xml (101 pag